# Grade 11 · Unit 3 — Bootstrap confidence intervals

> **SYNTHETIC** health table (invented); the method is real.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def data_file(dataset, filename):
    """Find datasets/<dataset>/clean/<filename> by walking up from this folder."""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        p = folder / "datasets" / dataset / "clean" / filename
        if p.exists():
            return p
    raise FileNotFoundError(f"Could not find {filename}. Open this notebook from inside the data-science-nepal folder.")

h = pd.read_csv(data_file("health-synthetic", "health_synthetic.csv"))
print(h.shape)

## One sample, one estimate
**Predict:** if we took another sample of 100 records, would the mean birth weight be exactly the same?

In [ ]:
rng = np.random.default_rng(0)
sample = h["birth_weight_kg"].sample(100, random_state=0).to_numpy()
print("sample mean:", round(sample.mean(), 3))

## Resample with replacement

In [ ]:
def bootstrap_means(x, B=2000, seed=0):
    r = np.random.default_rng(seed)
    return np.array([r.choice(x, size=len(x), replace=True).mean() for _ in range(B)])

boot = bootstrap_means(sample)
lo, hi = np.percentile(boot, [2.5, 97.5])
assert lo < sample.mean() < hi
print(f"95% bootstrap interval for the mean: {lo:.3f} to {hi:.3f} kg")
plt.hist(boot, bins=30, edgecolor="black"); plt.axvline(lo); plt.axvline(hi); plt.xlabel("bootstrap means (kg)"); plt.show()

## Bigger sample, narrower interval?

In [ ]:
widths = {}
for n in (25, 100, 400):
    x = h["birth_weight_kg"].sample(n, random_state=1).to_numpy()
    b = bootstrap_means(x, B=1000)
    widths[n] = np.subtract(*np.percentile(b, [97.5, 2.5]))
print({k: round(float(v), 3) for k, v in widths.items()})
assert widths[400] < widths[25]

**Make:** write a correct one-sentence interpretation of the interval — and one thing it does *not* say.